# HDB Rentals: block premiums, market level and what else works

Builds on `09`'s tuned LightGBM (holdout 487.5, public leaderboard 443.2) using only the competition and
auxiliary data. **Nine experiments in two phases**, each changing one thing, plus three submission checkpoints.
Two extras that use test-side information are kept in appendices and are not part of the model.

## Protocol (every experiment)

- **`09`'s two splits.** Tuning split: fit 2021-01 to 2023-03, score 2023-04 to 2023-12. Holdout: fit 2021-01 to
  2023-12, score 2024-01 to 2025-03.
- **Choose on the tuning split, report the holdout.** A change is kept only if it lowers the tuning RMSE against
  its reference. Holdout numbers are shown for every row but never pick anything.
- **No leakage.** Premiums, neighbour features and level forecasts are built from each split's fit rows only.
  Leaderboard scores are reported, never used to choose (except Appendix B, which is labelled as such).
- **Each experiment opens with a card** (question, reference, change, variants, what is held fixed, decision
  rule) and closes with its findings and a **Decision** line.

## Experiment log

Δ = RMSE minus the reference's RMSE on the same split; negative is better.

| ID | Experiment | Change | Reference | Tuning Δ | Holdout Δ | Decision |
|---|---|---|---|---|---|---|
| | **Phase 1: features on `09`'s model** | | | | | |
| E0 | Reference | `09`'s tuned LightGBM, unchanged | – | 603.6 | 487.5 | – |
| D1 | Block premium ceiling (diagnostic) | Actual market level × premium at four levels of detail | – | – | 491.6 at best | Expect a few dollars from E1, not tens |
| **E1** | Block + flat type premium | + block, block + flat type premiums and row count; 3 ways to build them | E0 | −3.8 (`oof`) | −4.4 | **Keep `oof`** |
| E2 | Coordinates | + km east/north, optionally rotated copies | E0 | −0.5 | +0.2 | Passes, then dropped in E5 |
| E3 | Neighbouring blocks | + premium of the 10 nearest blocks | E0 | −1.2 | −1.0 | Passes, then dropped in E5 |
| E4 | Recency weighting | Sample weights, half-life 6 to 36 months | E0 | +1.2 to +5.9 | +1.2 to +6.6 | Drop |
| E5 | Combine what passed | E1 + E2 + E3 together | E0 | −3.0 | −4.2 | **E1 alone is final** (beats the combination) |
| S1 | Submission | E5's model refit on all of train | – | – | – | `submission_10_features.csv`: leaderboard **436.8** (from 443.2) |
| | **Phase 2: market level and refinements** | | | | | |
| **E6** | Market level × premium | Time from a forecast market level instead of `MONTH_INDEX` | E5 (599.8 / 483.1) | −5.8 | −1.5 | **Keep `blend` forecast** |
| S2 | Submission | E6 alone, refit on all of train | – | – | – | `submission_10_level.csv`: no score recorded |
| E7 | Premium smoothing | Pull strength of the block premiums | E6 (smoothing 20 / 5) | −0.1 | −0.1 | Keep 5 / 2 (noise-sized) |
| E8 | More premiums | + street, town, flat model or per-sqm premium | E7 | −1.0 | +0.1 | Keep street + flat type |
| E9 | Retune LightGBM | 20 random draws on E6-E8's columns and target | E8 (holdout 481.7) | n/a* | −0.9 | Keep retuned |
| S3 | Final submission | E6 + E7 + E8 + E9, refit on all of train | – | – | – | `submission_10_final.csv`: leaderboard 439.5 (worse than S1) |
| | **Appendices: use test-side information, not valid models** | | | | | |
| A | Look up test rents | data.gov.sg's published records, which contain the test labels | – | – | 188.4 on the check months | `submission_10_external.csv` |
| B | Mix S1 and S3 | Weight computed from their two leaderboard scores | – | – | – | `submission_10_mix.csv`: leaderboard ≈ 434.5 |

\* E9's tuning RMSE is early-stopped on the tuning split itself, so it is not comparable with E8's.

**Contents.** Setup · E0 harness · Phase 1: D1, E1-E5, S1 · Phase 2: E6, S2, E7-E9, S3 · Summary · Appendix A ·
Appendix B

In [ ]:
import json
import time
import urllib.request
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA, EXTERNAL, OUT = ROOT / "data", ROOT / "data" / "external", ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 0
T0 = time.time()

## Setup

Cleaning, `DIST` and the two splits exactly as in `09` (§1 to §3); block coordinates come from the same HDB block
table. Column names are upper case, as in `09`.

- **Holdout:** fit 2021-01 to 2023-12, score 2024-01 to 2025-03.
- **Tuning split:** fit 2021-01 to 2023-03, score 2023-04 to 2023-12.
- **Model:** `09`'s tuned LightGBM, 357 rounds, the same on both splits.
- **Relative rent** = rent ÷ that month's median rent for the flat type. It takes the market level out, so a
  premium of 1.08 means "8% above the typical flat of this type that month". Premiums and neighbour features are
  built only from each split's fit rows; scored rows' rents never enter a feature.

In [ ]:
from scipy import sparse
from sklearn.neighbors import BallTree

CLEANED = ROOT / "cleaned"
TARGET = "MONTHLY_RENT"
KEY = ["TOWN", "BLOCK", "STREET"]


def lower_strip(df):
    out = df.copy()
    text = [c for c in out.columns if pd.api.types.is_string_dtype(out[c])]
    out[text] = out[text].apply(lambda s: s.str.strip().str.lower())
    return out


def clean_rent(df):
    out = lower_strip(df).drop(columns=["FURNISHED", "FEE"])
    out["FLAT_TYPE"] = out["FLAT_TYPE"].str.replace(r"[\s\-]+", "-", regex=True)
    d = pd.to_datetime(out["RENT_APPROVAL_DATE"], format="%Y-%m")
    out["RENT_APPROVAL_DATE"] = d
    out["MONTH_INDEX"] = (d.dt.year - 2021) * 12 + d.dt.month - 1
    out["FLAT_AGE"] = d.dt.year - out["LEASE_COMMENCE_DATE"]  # for the hedonic level in E6
    return out


def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


tr = clean_rent(pd.read_csv(DATA / "train.csv")).sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)
te = clean_rent(pd.read_csv(DATA / "test.csv"))

hdb = pd.read_csv(CLEANED / "sg_hdb_block_cleaned.csv", dtype={"POSTAL_CODE": str})
mrt = pd.read_csv(CLEANED / "sg_mrt_stations_cleaned.csv")
schools = pd.read_csv(CLEANED / "sg_schools_cleaned.csv")
malls = lower_strip(pd.read_csv(DATA / "auxiliary" / "sg-shopping-malls.csv"))
with open(DATA / "auxiliary" / "HawkerCentresGEOJSON.geojson") as f:
    geo = json.load(f)["features"]
hawker = lower_strip(pd.DataFrame({
    "NAME": [g["properties"]["NAME"] for g in geo],
    "STATUS": [g["properties"]["STATUS"] for g in geo],
    "LONGITUDE": [g["geometry"]["coordinates"][0] for g in geo],
    "LATITUDE": [g["geometry"]["coordinates"][1] for g in geo],
}))
mrt_open = mrt[mrt["STATUS"].eq("open")].drop_duplicates("NAME")
POI_GROUPS = {
    "primary": schools[schools["MAINLEVEL_CODE"].str.contains("primary", na=False)],
    "secondary": schools[schools["MAINLEVEL_CODE"].str.contains("secondary", na=False)],
    "mrt": mrt_open,
    "mrt_interchange": mrt_open[mrt_open["IS_INTERCHANGE"]],
    "malls": malls,
    "hawker": hawker[hawker["STATUS"].ne("under construction")],
}
lat, lon = hdb["LATITUDE"].to_numpy(), hdb["LONGITUDE"].to_numpy()
for prefix, poi in POI_GROUPS.items():
    d = haversine_km(lat[:, None], lon[:, None], poi["LATITUDE"].to_numpy()[None, :], poi["LONGITUDE"].to_numpy()[None, :])
    hdb[f"{prefix}_nearest_km"] = d.min(axis=1)
    for label, r in {"1km": 1.0, "2km": 2.0}.items():
        hdb[f"{prefix}_within_{label}"] = (d <= r).sum(axis=1)
cbd = mrt.loc[mrt["NAME"].eq("raffles place mrt station"), ["LATITUDE", "LONGITUDE"]].iloc[0]
hdb["dist_to_cbd_km"] = haversine_km(lat, lon, cbd["LATITUDE"], cbd["LONGITUDE"])

DIST = [
    "MAX_FLOOR", "primary_within_2km", "mrt_within_2km",
    "primary_nearest_km", "mrt_nearest_km", "secondary_within_1km",
    "mrt_interchange_within_1km", "mrt_interchange_nearest_km",
    "malls_within_1km", "malls_nearest_km", "hawker_within_1km", "hawker_nearest_km", "dist_to_cbd_km",
]
n_tr, n_te = len(tr), len(te)
tr = tr.merge(hdb[KEY + DIST + ["LATITUDE", "LONGITUDE"]], on=KEY, how="left", validate="m:1")
te = te.merge(hdb[KEY + DIST + ["LATITUDE", "LONGITUDE"]], on=KEY, how="left", validate="m:1")
assert len(tr) == n_tr and len(te) == n_te and tr["LATITUDE"].notna().all() and te["LATITUDE"].notna().all()

In [ ]:
# One id per block (town + block + street) and per block + flat type, shared by train and test.
BLOCKS = pd.concat([tr, te])[KEY + ["LATITUDE", "LONGITUDE"]].drop_duplicates(KEY).reset_index(drop=True)
BLOCKS["BID"] = np.arange(len(BLOCKS))
FT_LEVELS = sorted(tr["FLAT_TYPE"].unique())
NB, NFT = len(BLOCKS), len(FT_LEVELS)


def add_ids(df):
    out = df.merge(BLOCKS[KEY + ["BID"]], on=KEY, how="left", validate="m:1")
    out["FT"] = pd.Categorical(out["FLAT_TYPE"], categories=FT_LEVELS).codes
    out["G"] = out["BID"] * NFT + out["FT"]
    return out


tr, te = add_ids(tr), add_ids(te)
assert (te["FT"] >= 0).all()
tr["REL"] = tr[TARGET] / tr.groupby(["MONTH_INDEX", "FLAT_TYPE"])[TARGET].transform("median")
# The same per square metre, for E8's per-sqm premium.
rent_sqm = tr[TARGET] / tr["FLOOR_AREA_SQM"]
tr["REL_SQM"] = rent_sqm / rent_sqm.groupby([tr["MONTH_INDEX"], tr["FLAT_TYPE"]]).transform("median")

BASE_NUM = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "MONTH_INDEX"] + DIST
CAT = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
CATEGORIES = {c: pd.CategoricalDtype(sorted(tr[c].unique())) for c in CAT}
LGBM_TUNED = {"num_leaves": 63, "min_child_samples": 100, "subsample": 0.85, "subsample_freq": 1,
              "colsample_bytree": 0.5, "reg_lambda": 20.0, "cat_smooth": 30.0, "n_estimators": 357,
              "learning_rate": 0.05}


def matrix(df, extra=(), time=True):
    num = BASE_NUM if time else [c for c in BASE_NUM if c != "MONTH_INDEX"]
    X = df[num + list(extra) + CAT].copy()
    for c in CAT:
        X[c] = X[c].astype(CATEGORIES[c])
    return X


def fit_lgbm(fit_df, extra=(), weight=None, **overrides):
    est = lgb.LGBMRegressor(**{**LGBM_TUNED, **overrides}, verbose=-1, n_jobs=-1, random_state=SEED)
    return est.fit(matrix(fit_df, extra), fit_df[TARGET].to_numpy(), sample_weight=weight)


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(p, float)) ** 2)))


SPLITS = {"tuning": ("2023-04", "2024-01"), "holdout": ("2024-01", "2099-01")}
FRAMES = {}
for s, (start, end) in SPLITS.items():
    d = tr["RENT_APPROVAL_DATE"]
    FRAMES[s] = [tr[d < start].reset_index(drop=True), tr[(d >= start) & (d < end)].reset_index(drop=True)]
    print(f"{s:<8} fit {len(FRAMES[s][0]):>7,} rows, score {len(FRAMES[s][1]):>6,} rows")
print(f"blocks: {NB:,}  block + flat type pairs seen in train: {tr['G'].nunique():,}")

tuning   fit  78,960 rows, score 26,877 rows
holdout  fit 105,837 rows, score 44,163 rows
blocks: 9,456  block + flat type pairs seen in train: 14,828


## E0. Experiment harness and reference

`run(name, extra, half_life)` fits the model on each split's fit rows with the base columns plus `extra` (and
optional recency weights) and records tuning and holdout RMSE and bias in `RESULTS`. `results_table` shows rows
against **E0**, `09`'s tuned LightGBM with no extra columns. Phase 1 (D1 to E5) uses this harness.

In [ ]:
RESULTS = {}


def run(name, extra=(), half_life=None):
    row = {}
    for s, (fit_df, val_df) in FRAMES.items():
        m = fit_df["MONTH_INDEX"].to_numpy()
        w = None if half_life is None else 0.5 ** ((m.max() - m) / half_life)
        p = fit_lgbm(fit_df, extra, w).predict(matrix(val_df, extra))
        y = val_df[TARGET].to_numpy()
        row[f"{s} RMSE"], row[f"{s} bias"] = rmse(y, p), float(np.mean(p - y))
    RESULTS[name] = {**row, "extra": list(extra), "half_life": half_life}
    return row


def results_table(names):
    t = pd.DataFrame({n: RESULTS[n] for n in names}).T[["tuning RMSE", "holdout RMSE", "tuning bias", "holdout bias"]]
    t = t.astype(float)
    for s in ["tuning", "holdout"]:
        t[f"{s} vs base"] = t[f"{s} RMSE"] - RESULTS["base (09 tuned LightGBM)"][f"{s} RMSE"]
    return t[["tuning RMSE", "tuning vs base", "holdout RMSE", "holdout vs base", "tuning bias", "holdout bias"]]


run("base (09 tuned LightGBM)")
results_table(["base (09 tuned LightGBM)"])

,tuning RMSE,tuning vs base,holdout RMSE,holdout vs base,tuning bias,holdout bias
base (09 tuned LightGBM),603.60,0.00,487.49,0.00,-89.29,-35.03


# Phase 1: features on top of `09`'s model

Reference for every experiment in this phase: **E0** (tuning 603.6, holdout 487.5).

## D1. How much can a block's own past rents tell? (diagnostic)

| | |
|---|---|
| **Question** | Before building any feature: if the market level were known exactly, how good is a premium at each level of detail? |
| **Method** | Predict each holdout row as (actual median rent for its month and flat type) × (premium from the fit months) |
| **Varied** | Premium detail: none · town + flat type · block · block + flat type |
| **Caveat** | Uses the holdout's actual level, so these are not achievable scores. They isolate the premium part and show which level of detail carries information |
| **Used for** | Setting expectations for E1 |

Premiums are smoothed averages of relative rent: a group with few rows is pulled toward the level above it
(block + flat type toward block with weight 5 rows, block toward 1 with weight 20, town + flat type toward 1 with
weight 20).

In [5]:
M_BLOCK, M_FT = 20, 5


def encode(src, dst, w, m=(M_BLOCK, M_FT)):
    # Smoothed block and block + flat type premiums from src rows (weights w), looked up for dst rows.
    m_block, m_ft = m
    bid, g, rel = src["BID"].to_numpy(), src["G"].to_numpy(), src["REL"].to_numpy()
    sb, cb = np.bincount(bid, w * rel, NB), np.bincount(bid, w, NB)
    sg, cg = np.bincount(g, w * rel, NB * NFT), np.bincount(g, w, NB * NFT)
    n = np.bincount(g, minlength=NB * NFT)
    db, dg = dst["BID"].to_numpy(), dst["G"].to_numpy()
    pb = (m_block + sb[db]) / (m_block + cb[db])
    pg = (m_ft * pb + sg[dg]) / (m_ft + cg[dg])
    return np.column_stack([pb, pg, n[dg]])


fit_h, hold_h = FRAMES["holdout"]
y_h = hold_h[TARGET].to_numpy()
oracle_level = hold_h.groupby(["MONTH_INDEX", "FLAT_TYPE"])[TARGET].transform("median").to_numpy()
town_ft = fit_h.groupby(["TOWN", "FLAT_TYPE"])["REL"].agg(["sum", "size"])
town_ft_prem = ((M_BLOCK + town_ft["sum"]) / (M_BLOCK + town_ft["size"])).rename("p")
p_town = hold_h.join(town_ft_prem, on=["TOWN", "FLAT_TYPE"])["p"].fillna(1).to_numpy()
p_block, p_block_ft, n_seen = encode(fit_h, hold_h, np.ones(len(fit_h))).T
seen = n_seen > 0

check_tbl = pd.DataFrame({
    "flat type only (premium = 1)": [rmse(y_h, oracle_level)],
    "town + flat type": [rmse(y_h, oracle_level * p_town)],
    "block": [rmse(y_h, oracle_level * p_block)],
    "block + flat type": [rmse(y_h, oracle_level * p_block_ft)],
}, index=["holdout RMSE with the actual level"]).T
check_tbl["block + flat type seen in fit (" + f"{seen.mean():.0%}" + " of rows)"] = [
    rmse(y_h[seen], (oracle_level * p)[seen]) for p in [np.ones(len(y_h)), p_town, p_block, p_block_ft]]
display(check_tbl)
print("for reference, 09's tuned LightGBM on the holdout: 487.5 (it has to forecast the level too)")

,holdout RMSE with the actual level,block + flat type seen in fit (99% of rows)
flat type only (premium = 1),567.71,566.63
town + flat type,516.01,514.51
block,511.00,509.06
block + flat type,491.58,489.27


for reference, 09's tuned LightGBM on the holdout: 487.5 (it has to forecast the level too)


**D1 findings**

- **Even with the actual market level, the block + flat type premium only reaches 491.6**, about where `09`'s model already is (487.5) while also having to forecast the level. Detail helps in steps (flat type only 567.7, town + flat type 516.0, block 511.0, block + flat type 491.6), but the last step is worth about 25, and the model's town, flat model, floor area and `DIST` columns already cover much of it.
- **99% of holdout rows have their block + flat type in the fit months**, so this is not a coverage problem: the same block's flats of one type just rent at quite different prices from one year to the next.

**Takeaway:** block history is not worth 100+ RMSE. Expect a gain of a few dollars from E1, not tens.

## E1. Block + flat type premium

| | |
|---|---|
| **Question** | Does a block's rent history, relative to the market, add information the model lacks? |
| **Reference** | E0 |
| **Change** | + 3 columns: block premium, block + flat type premium, and how many fit rows the block + flat type had (so the tree can tell a premium from 30 rentals from one built on 1) |
| **Variants** | How each fit row gets its own premium: `oof` · `past` · `recent` (below) |
| **Held fixed** | Model settings (357 rounds), base columns, smoothing 20 / 5, splits. Scored and test rows always use all fit months |
| **Decision rule** | Keep the variant with the lowest tuning RMSE if it beats E0 |

The variants differ only in what the **fit rows** see:

- **`oof` (out-of-fold):** each fit row's premium comes from the other 4/5 of the fit rows, over all months (as
  `08` did for blocks).
- **`past` (past only):** each fit row's premium comes only from earlier months. This is how a test row sees the
  data, so the tree learns how much to trust a premium as it would be at prediction time.
- **`recent` (past only, recency-weighted):** as `past`, with each earlier rental's weight halving every 12 months,
  so the premium follows a block that is drifting.

In [6]:
PREMIUM_VARIANTS = {"oof": ("oof", None), "past": ("past", None), "recent": ("past", 12)}


def premiums(fit_df, apply_dfs, mode, half_life=None, m=(M_BLOCK, M_FT)):
    month = fit_df["MONTH_INDEX"].to_numpy()
    weights = lambda m, ref: np.ones(len(m)) if half_life is None else 0.5 ** ((ref - m) / half_life)
    T = month.max()
    applied = [encode(fit_df, a, weights(month, T), m) for a in apply_dfs]
    own = np.empty((len(fit_df), 3))
    if mode == "oof":
        for f_idx, p_idx in KFold(5, shuffle=True, random_state=SEED).split(fit_df):
            src = fit_df.iloc[f_idx]
            own[p_idx] = encode(src, fit_df.iloc[p_idx], weights(month[f_idx], T), m)
    else:
        for t in np.unique(month):
            before, now = month < t, month == t
            own[now] = encode(fit_df[before], fit_df[now], weights(month[before], t - 1), m)
    return own, applied


def premium_cols(tag):
    return [f"BLOCK_PREM_{tag}", f"BLOCK_FT_PREM_{tag}", f"BLOCK_FT_N_{tag}"]


for s, (fit_df, val_df) in FRAMES.items():
    for tag, (mode, hl) in PREMIUM_VARIANTS.items():
        own, (applied,) = premiums(fit_df, [val_df], mode, hl)
        fit_df[premium_cols(tag)], val_df[premium_cols(tag)] = own, applied

for tag in PREMIUM_VARIANTS:
    run(f"+ block premiums, {tag}", premium_cols(tag))
results_table(["base (09 tuned LightGBM)"] + [f"+ block premiums, {t}" for t in PREMIUM_VARIANTS])

,tuning RMSE,tuning vs base,holdout RMSE,holdout vs base,tuning bias,holdout bias
base (09 tuned LightGBM),603.60,0.00,487.49,0.00,-89.29,-35.03
"+ block premiums, oof",599.79,-3.81,483.14,-4.35,-93.86,-37.08
"+ block premiums, past",601.12,-2.48,483.32,-4.17,-93.64,-34.73
"+ block premiums, recent",601.87,-1.73,485.43,-2.06,-93.35,-36.78


**E1 findings**

- **The out-of-fold premiums help a little on both splits:** tuning 603.6 → 599.8 (−3.8), holdout 487.5 → 483.1 (−4.4). It is the largest gain any single change has made since the report's `DIST` set.
- **Past only is close behind** (−2.5 / −4.2). **Recency weighting the premium is worse than either** (−1.7 / −2.1): with few rentals per block + flat type, down-weighting older ones mostly adds noise.
- Bias barely moves (−94 vs −89 on tuning, −37 vs −35 on holdout): premiums sharpen who is above or below the market, not where the market goes.

**Decision: keep `oof`.** It goes to E5.

## E2. Coordinates

| | |
|---|---|
| **Question** | Does raw position help beyond `TOWN` and `DIST`, including along diagonal boundaries? |
| **Reference** | E0 |
| **Change** | + `X_KM`, `Y_KM`: each block's position in km east and north of the island's centre |
| **Variants** | plain (2 columns) · rotated (+ copies rotated by 15°, 30°, 45°, 60° and 75°: 7 columns) |
| **Held fixed** | As E1 |
| **Decision rule** | Keep the variant with the lowest tuning RMSE if it beats E0 |

Why rotated copies: trees split one column at a time, so a diagonal boundary (a river, an expressway) needs many
splits; the rotated columns give the tree those diagonals directly.

In [7]:
LAT0, LON0 = 1.35, 103.82
BLOCKS["X_KM"] = (BLOCKS["LONGITUDE"] - LON0) * 111.32 * np.cos(np.radians(LAT0))
BLOCKS["Y_KM"] = (BLOCKS["LATITUDE"] - LAT0) * 110.57
ANGLES = [15, 30, 45, 60, 75]
for a in ANGLES:
    r = np.radians(a)
    BLOCKS[f"ROT_{a}"] = BLOCKS["X_KM"] * np.cos(r) + BLOCKS["Y_KM"] * np.sin(r)
COORDS = ["X_KM", "Y_KM"]
ROTATED = COORDS + [f"ROT_{a}" for a in ANGLES]

for fit_df, val_df in FRAMES.values():
    for df in (fit_df, val_df):
        df[ROTATED] = BLOCKS.loc[df["BID"], ROTATED].to_numpy()

run("+ coordinates", COORDS)
run("+ coordinates, rotated", ROTATED)
results_table(["base (09 tuned LightGBM)", "+ coordinates", "+ coordinates, rotated"])

,tuning RMSE,tuning vs base,holdout RMSE,holdout vs base,tuning bias,holdout bias
base (09 tuned LightGBM),603.60,0.00,487.49,0.00,-89.29,-35.03
+ coordinates,603.12,-0.48,487.64,0.15,-89.46,-32.12
"+ coordinates, rotated",603.41,-0.19,487.74,0.25,-92.50,-33.01


**E2 findings**

- **Coordinates do nothing:** −0.5 on tuning, +0.2 on holdout; rotated copies are no better (−0.2 / +0.3).
  `TOWN` and the 13 `DIST` columns already pin down location.

**Decision:** plain coordinates pass the rule (by 0.5 on tuning), so they go to E5; rotated copies do not.

## E3. Neighbouring blocks

| | |
|---|---|
| **Question** | Do nearby blocks' rents say something about a block's own? (The cheap version of a graph model: "blocks near each other rent alike") |
| **Reference** | E0 |
| **Change** | + 2 columns: the pooled premium of the 10 nearest other blocks for the row's flat type, and over all flat types, smoothed as in E1 |
| **Variants** | neighbours alone · neighbours + E2's rotated coordinates |
| **Held fixed** | As E1. Built from fit rows only, and a row's own block is never in its neighbour set, so its own rent never enters |
| **Decision rule** | Keep if it beats E0 on the tuning split |

In [8]:
K_NEIGHBOURS = 10
pos = np.radians(BLOCKS[["LATITUDE", "LONGITUDE"]].to_numpy())
dist, idx = BallTree(pos, metric="haversine").query(pos, k=K_NEIGHBOURS + 1)
src_rows = np.repeat(np.arange(NB), K_NEIGHBOURS + 1)
keep = idx.ravel() != src_rows
ADJ = sparse.csr_matrix((np.ones(keep.sum()), (src_rows[keep], idx.ravel()[keep])), shape=(NB, NB))
print(f"median distance to the 10th nearest block: {np.median(dist[:, -1]) * 6371:.2f} km")
NEIGHBOUR = ["NBR_FT_PREM", "NBR_PREM"]


def neighbour_premiums(fit_df, dfs):
    g, rel = fit_df["G"].to_numpy(), fit_df["REL"].to_numpy()
    sg = np.bincount(g, rel, NB * NFT).reshape(NB, NFT)
    cg = np.bincount(g, minlength=NB * NFT).reshape(NB, NFT).astype(float)
    ns, nc = ADJ @ sg, ADJ @ cg
    p_all = (M_BLOCK + ns.sum(1)) / (M_BLOCK + nc.sum(1))
    p_ft = (M_FT * p_all[:, None] + ns) / (M_FT + nc)
    return [np.column_stack([p_ft[d["BID"].to_numpy(), d["FT"].to_numpy()], p_all[d["BID"].to_numpy()]]) for d in dfs]


for fit_df, val_df in FRAMES.values():
    fit_df[NEIGHBOUR], val_df[NEIGHBOUR] = neighbour_premiums(fit_df, [fit_df, val_df])

run("+ neighbours", NEIGHBOUR)
run("+ coordinates, rotated + neighbours", ROTATED + NEIGHBOUR)
results_table(["base (09 tuned LightGBM)", "+ neighbours", "+ coordinates, rotated + neighbours"])

median distance to the 10th nearest block: 0.17 km


,tuning RMSE,tuning vs base,holdout RMSE,holdout vs base,tuning bias,holdout bias
base (09 tuned LightGBM),603.60,0.00,487.49,0.00,-89.29,-35.03
+ neighbours,602.40,-1.20,486.45,-1.04,-93.63,-32.50
"+ coordinates, rotated + neighbours",602.68,-0.92,486.17,-1.32,-92.85,-32.65


**E3 findings**

- **Neighbours help slightly and consistently:** −1.2 tuning, −1.0 holdout. The 10 nearest blocks are within about 170 m, so this is a very local premium. Adding rotated coordinates on top does not help on the tuning split (−0.9).
- A graph network would learn the same "nearby blocks rent alike" signal; since the cheap version is worth about 1, a graph network is not worth building for this data.

**Decision:** neighbours alone pass and go to E5.

## E4. Weighting recent months

| | |
|---|---|
| **Question** | Does weighting recent months more help the model follow the market? |
| **Reference** | E0 |
| **Change** | Each training row is weighted 0.5^(months before the last fit month ÷ half-life). A half-life of 12 means a row from a year before the cutoff counts half as much as one from the last month |
| **Variants** | half-life 6, 12, 24, 36 months |
| **Held fixed** | Base columns (no premiums), settings, splits |
| **Decision rule** | Keep the best half-life if it beats E0 on the tuning split |

In [9]:
HALF_LIVES = [6, 12, 24, 36]
for hl in HALF_LIVES:
    run(f"weighted, half-life {hl}", half_life=hl)
results_table(["base (09 tuned LightGBM)"] + [f"weighted, half-life {hl}" for hl in HALF_LIVES])

,tuning RMSE,tuning vs base,holdout RMSE,holdout vs base,tuning bias,holdout bias
base (09 tuned LightGBM),603.60,0.00,487.49,0.00,-89.29,-35.03
"weighted, half-life 6",609.45,5.85,494.05,6.56,-90.69,-33.09
"weighted, half-life 12",606.30,2.70,490.30,2.81,-90.20,-34.02
"weighted, half-life 24",604.83,1.23,488.85,1.35,-89.98,-32.35
"weighted, half-life 36",605.05,1.45,488.68,1.19,-89.27,-33.96


**E4 findings**

- **Weighting recent months hurts at every half-life:** +5.9 / +6.6 at 6 months, still +1.2 / +1.2 at 24 to 36 months. Older rows still teach the tree how flats differ from each other, and dropping their weight loses that more than it helps follow the recent market.
- The bias does not improve either (−90 on tuning, −32 to −34 on holdout), so weighting does not fix the under-prediction far ahead.

**Decision: drop.** Nothing goes to E5.

## E5. Everything that passed, together

| | |
|---|---|
| **Question** | Do the changes that passed on their own add up? |
| **Reference** | E0, and each part alone |
| **Change** | From each family (E1 premium variant, E2 coordinates, E3 neighbours, E4 weighting), the variant with the lowest tuning RMSE, if it beats E0 there |
| **Decision rule** | The final Phase 1 model is whichever of the combination and its parts has the lowest tuning RMSE (combining can do worse than its best part) |

The second cell breaks the holdout down by months ahead for E0 and the final model.

In [10]:
FAMILIES = {
    "premium": [f"+ block premiums, {t}" for t in PREMIUM_VARIANTS],
    "coordinates": ["+ coordinates", "+ coordinates, rotated"],
    "neighbours": ["+ neighbours"],
    "weighting": [f"weighted, half-life {hl}" for hl in HALF_LIVES],
}
base_tune = RESULTS["base (09 tuned LightGBM)"]["tuning RMSE"]
chosen = {}
for fam, names in FAMILIES.items():
    best_name = min(names, key=lambda n: RESULTS[n]["tuning RMSE"])
    if RESULTS[best_name]["tuning RMSE"] < base_tune:
        chosen[fam] = best_name
print("kept:", chosen)

combo_extra = [c for n in chosen.values() for c in RESULTS[n]["extra"]]
combo_hl = RESULTS[chosen["weighting"]]["half_life"] if "weighting" in chosen else None
run("combined", combo_extra, combo_hl)
# Combining can do worse than its best part, so the final model is whichever is best on the tuning split.
final = min(["combined"] + list(chosen.values()), key=lambda n: RESULTS[n]["tuning RMSE"])
final_extra, final_hl = RESULTS[final]["extra"], RESULTS[final]["half_life"]
print("final model:", final)
results_table(["base (09 tuned LightGBM)"] + list(chosen.values()) + ["combined"])

kept: {'premium': '+ block premiums, oof', 'coordinates': '+ coordinates', 'neighbours': '+ neighbours'}
final model: + block premiums, oof


,tuning RMSE,tuning vs base,holdout RMSE,holdout vs base,tuning bias,holdout bias
base (09 tuned LightGBM),603.60,0.00,487.49,0.00,-89.29,-35.03
"+ block premiums, oof",599.79,-3.81,483.14,-4.35,-93.86,-37.08
+ coordinates,603.12,-0.48,487.64,0.15,-89.46,-32.12
+ neighbours,602.40,-1.20,486.45,-1.04,-93.63,-32.50
combined,600.63,-2.98,483.27,-4.23,-95.51,-37.72


In [11]:
# Error by months ahead on the holdout, base vs the final model.
fit_df, val_df = FRAMES["holdout"]
m = fit_df["MONTH_INDEX"].to_numpy()
w = None if final_hl is None else 0.5 ** ((m.max() - m) / final_hl)
p_base = fit_lgbm(fit_df).predict(matrix(val_df))
p_combo = fit_lgbm(fit_df, final_extra, w).predict(matrix(val_df, final_extra))
ahead = val_df["MONTH_INDEX"] - m.max()
bucket = pd.cut(ahead, [0, 3, 6, 9, 12, 15], labels=["1-3", "4-6", "7-9", "10-12", "13-15"])
y = val_df[TARGET].to_numpy()
pd.DataFrame({
    "base RMSE": pd.Series((p_base - y) ** 2).groupby(bucket, observed=True).mean() ** 0.5,
    "final RMSE": pd.Series((p_combo - y) ** 2).groupby(bucket, observed=True).mean() ** 0.5,
    "base bias": pd.Series(p_base - y).groupby(bucket, observed=True).mean(),
    "final bias": pd.Series(p_combo - y).groupby(bucket, observed=True).mean(),
}).rename_axis("months ahead")

,base RMSE,final RMSE,base bias,final bias
months ahead,,,,
1-3,527.50,525.33,4.85,1.98
4-6,506.00,500.68,-8.89,-8.90
7-9,482.00,477.64,-51.63,-53.26
10-12,453.42,447.97,-60.91,-63.04
13-15,461.23,456.41,-61.80,-65.43


**E5 findings**

- **Adding coordinates and neighbours on top of the premiums makes it worse on the tuning split** (600.6 vs 599.8).
- **The final model is better than E0 in every months-ahead bucket** (2 to 5 RMSE each), and the far-ahead under-prediction (−63 to −65 at 10 to 15 months) is untouched: that is the level problem from `07`/`08`, which no feature here addresses.

**Decision: the Phase 1 model is E0 + E1's `oof` block premiums.** It is the reference for Phase 2.

### S1. Submission checkpoint: E5's model

E5's model refit on all of train, with rounds scaled to the larger training set as `09` did
(357 × all train ÷ holdout fit). Train rows get the chosen premium variant; test rows use all of train. E0 is
refit alongside as a check that it reproduces `submission_09_lgbm.csv`.

In [12]:
# Refit on all of train. Premiums for train rows use the chosen variant; test rows use all of train.
full_tr, full_te = tr.copy(), te.copy()
for df in (full_tr, full_te):
    df[ROTATED] = BLOCKS.loc[df["BID"], ROTATED].to_numpy()
full_tr[NEIGHBOUR], full_te[NEIGHBOUR] = neighbour_premiums(full_tr, [full_tr, full_te])
for tag, (mode, hl) in PREMIUM_VARIANTS.items():
    own, (applied,) = premiums(full_tr, [full_te], mode, hl)
    full_tr[premium_cols(tag)], full_te[premium_cols(tag)] = own, applied

rounds = int(round(LGBM_TUNED["n_estimators"] * len(full_tr) / len(FRAMES["holdout"][0])))
m = full_tr["MONTH_INDEX"].to_numpy()
w = None if final_hl is None else 0.5 ** ((m.max() - m) / final_hl)
test_base = fit_lgbm(full_tr, n_estimators=rounds).predict(matrix(full_te))
test_combo = fit_lgbm(full_tr, final_extra, w, n_estimators=rounds).predict(matrix(full_te, final_extra))

prev = pd.read_csv(OUT / "submission_09_lgbm.csv")["Predicted"].to_numpy()
print(f"rounds on all of train: {rounds}")
print(f"base refit vs submission_09_lgbm.csv (should be ~0, same model): RMSE {rmse(prev, test_base):.2f}")
print(f"final vs base on test: RMSE between them {rmse(test_base, test_combo):.1f}, "
      f"mean shift {np.mean(test_combo - test_base):+.1f}")
assert len(test_combo) == len(te) and np.isfinite(test_combo).all()
SUB_FEATURES = OUT / "submission_10_features.csv"
pd.DataFrame({"Id": np.arange(len(test_combo)), "Predicted": test_combo}).to_csv(SUB_FEATURES, index=False)
print(f"wrote {SUB_FEATURES}")

rounds on all of train: 506
base refit vs submission_09_lgbm.csv (should be ~0, same model): RMSE 0.93
final vs base on test: RMSE between them 73.1, mean shift +2.3
wrote ..\outputs\submission_10_features.csv


**S1 notes**

- **File:** `outputs/submission_10_features.csv`, **public leaderboard 436.8** (from 443.2).
- The E0 refit reproduces `submission_09_lgbm.csv` to within 0.9 RMSE, and the new predictions move by 73 RMSE from it with almost no mean shift (+2.3). Locally that is worth about −4.

# Phase 2: market level and refinements

Reference for E6: **E5's model** (E0 + `oof` block premiums; tuning 599.8, holdout 483.1). From E7 on, each
experiment's reference is the previous experiment's chosen setup.

## E6. Market level × premium

| | |
|---|---|
| **Question** | Does forecasting the market level explicitly beat letting the tree hold the last level through `MONTH_INDEX`? |
| **Reference** | E5's model (time from `MONTH_INDEX`) |
| **Change** | rent ≈ market level(month) × premium(flat, location). LightGBM learns the premium; `MONTH_INDEX` is dropped |
| **Variants** | How the level is carried past the last fit month: `hold` · `damped` · `blend` |
| **Prior from `08`** | Only `blend` is eligible: at nine cutoffs `damped` blew up once (over-predicting by 533 right after the 2022 climb ended) and `blend` was best on both the mean and the worst cutoff. `hold` and `damped` are shown for information |
| **Held fixed** | E5's columns (base + `oof` block premiums), settings (357 rounds), splits |
| **Decision rule** | Keep `blend` if it beats `MONTH_INDEX` on the tuning split |

Why: a tree can't extrapolate `MONTH_INDEX`, so past the last training month it holds the last level, and on
the holdout it under-predicts by about 60 at 10 to 15 months ahead (E5). `07` §6 and `08` §4 built the
alternative; it is repeated here with E1's block premiums added.

- **Level:** month effects from an ordinary least squares fit of log rent on month dummies plus town, flat type,
  flat model, floor area and flat age, refit on each split's fit months only (`08` §4).
- **Premium:** LightGBM learns rent ÷ level, with `MONTH_INDEX` dropped. Rows are weighted by level², so the
  squared error it minimises is the squared error in dollars.
- **Forecast past the last fit month:** `hold` repeats the last level, `damped` carries a damped trend (Holt's
  method), `blend` averages the two in log space.

The first cell defines the level model and the Phase 2 harness (`fit_arm`, `run2`, `table2`, results in `R2`),
used by E6 to E9. The second runs E6; the third breaks the holdout down by months ahead.

In [13]:
import warnings

from statsmodels.tsa.holtwinters import ExponentialSmoothing

HEDONIC_CAT, HEDONIC_NUM = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"], ["FLOOR_AREA_SQM", "FLAT_AGE"]


def hedonic_level(df):
    # Log market level per MONTH_INDEX, relative to the first month (08 §4).
    months = df["MONTH_INDEX"]
    X = pd.concat([
        pd.get_dummies(months, prefix="m", drop_first=True, dtype=float),
        pd.get_dummies(df[HEDONIC_CAT], drop_first=True, dtype=float),
        df[HEDONIC_NUM].astype(float),
    ], axis=1)
    X.insert(0, "const", 1.0)
    coef, *_ = np.linalg.lstsq(X.to_numpy(), np.log(df[TARGET].to_numpy()), rcond=None)
    coef = pd.Series(coef, index=X.columns)
    ordered = np.sort(months.unique())
    return pd.Series([0.0] + [coef[f"m_{m}"] for m in ordered[1:]], index=ordered)


def forecast_level(level, months, method):
    ahead = np.sort(np.unique(months[months > level.index.max()]))
    hold = np.full(len(ahead), level.iloc[-1])
    if method == "hold":
        future = hold
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = ExponentialSmoothing(level.to_numpy(), trend="add", damped_trend=True,
                                       initialization_method="estimated").fit()
        damped = fit.forecast(len(ahead))
        future = damped if method == "damped" else (hold + damped) / 2
    return pd.concat([level, pd.Series(future, index=ahead)])


def level_scales(fit_df, apply_dfs, method):
    months = np.concatenate([a["MONTH_INDEX"].to_numpy() for a in apply_dfs])
    level = forecast_level(hedonic_level(fit_df), months, method)
    to_scale = lambda df: np.exp(level.reindex(df["MONTH_INDEX"]).to_numpy())
    return to_scale(fit_df), [to_scale(a) for a in apply_dfs]


def fit_arm(fit_df, apply_dfs, extra, arm, params=None, early_stop=False):
    # Fit LightGBM on rent with MONTH_INDEX ("MONTH_INDEX") or on rent / level ("level-<method>"), and predict
    # rent for each frame in apply_dfs. With early_stop, apply_dfs[0] is the early-stopping set and the
    # number of rounds used is returned too.
    params = {**LGBM_TUNED, **(params or {})}
    time_col = arm == "MONTH_INDEX"
    if time_col:
        s_fit, s_apply = np.ones(len(fit_df)), [np.ones(len(a)) for a in apply_dfs]
    else:
        s_fit, s_apply = level_scales(fit_df, apply_dfs, arm.split("-", 1)[1])
    est = lgb.LGBMRegressor(**params, verbose=-1, n_jobs=-1, random_state=SEED)
    kw = {}
    if early_stop:
        e = apply_dfs[0]
        kw = {"eval_X": matrix(e, extra, time_col), "eval_y": e[TARGET].to_numpy() / s_apply[0],
              "eval_sample_weight": [s_apply[0] ** 2], "callbacks": [lgb.early_stopping(100, verbose=False)]}
    est.fit(matrix(fit_df, extra, time_col), fit_df[TARGET].to_numpy() / s_fit, sample_weight=s_fit ** 2, **kw)
    preds = [est.predict(matrix(a, extra, time_col)) * s for a, s in zip(apply_dfs, s_apply)]
    return (preds, est.best_iteration_) if early_stop else preds


R2 = {}


def run2(name, extra, arm, params=None):
    row = {}
    for s, (fit_df, val_df) in FRAMES.items():
        (p,) = fit_arm(fit_df, [val_df], extra, arm, params)
        y = val_df[TARGET].to_numpy()
        row[f"{s} RMSE"], row[f"{s} bias"] = rmse(y, p), float(np.mean(p - y))
    R2[name] = {**row, "extra": list(extra), "arm": arm}
    return row


def table2(names, ref):
    t = pd.DataFrame({n: R2[n] for n in names}).T[["tuning RMSE", "holdout RMSE", "tuning bias", "holdout bias"]].astype(float)
    for s in ["tuning", "holdout"]:
        t[f"{s} vs ref"] = t[f"{s} RMSE"] - R2[ref][f"{s} RMSE"]
    return t[["tuning RMSE", "tuning vs ref", "holdout RMSE", "holdout vs ref", "tuning bias", "holdout bias"]]

In [14]:
ARMS = ["MONTH_INDEX", "level-hold", "level-damped", "level-blend"]
for arm in ARMS:
    run2(f"E5 final, {arm}", final_extra, arm)
REF = "E5 final, MONTH_INDEX"
ARM = "level-blend" if R2["E5 final, level-blend"]["tuning RMSE"] < R2[REF]["tuning RMSE"] else "MONTH_INDEX"
print(f"setup carried forward: {ARM}  (reference row = E5's model, S1 leaderboard 436.8)")
table2([f"E5 final, {a}" for a in ARMS], REF)

setup carried forward: level-blend  (reference row = E5's model, S1 leaderboard 436.8)


,tuning RMSE,tuning vs ref,holdout RMSE,holdout vs ref,tuning bias,holdout bias
"E5 final, MONTH_INDEX",599.79,0.00,483.14,0.00,-93.86,-37.08
"E5 final, level-hold",601.47,1.68,482.60,-0.54,-109.41,-19.57
"E5 final, level-damped",591.19,-8.60,482.11,-1.03,-12.49,26.49
"E5 final, level-blend",593.95,-5.84,481.65,-1.49,-61.42,3.36


In [15]:
# Error by months ahead on the holdout, MONTH_INDEX vs blend.
fit_df, val_df = FRAMES["holdout"]
(p_mi,) = fit_arm(fit_df, [val_df], final_extra, "MONTH_INDEX")
(p_bl,) = fit_arm(fit_df, [val_df], final_extra, "level-blend")
y = val_df[TARGET].to_numpy()
bucket = pd.cut(val_df["MONTH_INDEX"] - fit_df["MONTH_INDEX"].max(), [0, 3, 6, 9, 12, 15],
                labels=["1-3", "4-6", "7-9", "10-12", "13-15"])
grp = lambda v: pd.Series(v).groupby(bucket, observed=True)
pd.DataFrame({"MONTH_INDEX RMSE": grp((p_mi - y) ** 2).mean() ** 0.5, "blend RMSE": grp((p_bl - y) ** 2).mean() ** 0.5,
              "MONTH_INDEX bias": grp(p_mi - y).mean(), "blend bias": grp(p_bl - y).mean()}).rename_axis("months ahead")

,MONTH_INDEX RMSE,blend RMSE,MONTH_INDEX bias,blend bias
months ahead,,,,
1-3,525.33,526.52,1.98,24.24
4-6,500.68,501.60,-8.90,23.18
7-9,477.64,475.34,-53.26,-11.43
10-12,447.97,443.97,-63.04,-12.50
13-15,456.41,452.17,-65.43,-8.85


**E6 findings**

- **The blend beats `MONTH_INDEX` on both splits:** tuning 599.8 → 594.0 (−5.8), holdout 483.1 → 481.7 (−1.5).
- **It fixes the far-ahead under-prediction:** bias at 10 to 15 months goes from −63/−65 to −13/−9, and RMSE there drops 4. The price is a small over-prediction in the first 6 months (+23/+24 bias vs about 0), which is why the overall holdout gain is only 1.5.
- **Damped alone looks best on the tuning split (−8.6)** but over-predicts on the holdout (+26 bias); `08` showed it can over-predict by 500+ right after a climb ends. Hold changes little. The blend sits between them on both splits, as in `08`.

**Decision: keep the `blend` level.** It is the setup for E7 to E9.

### S2. Submission checkpoint: E6 alone

E6's setup on E5's columns, refit on all of train, so the leaderboard can tell the level change apart from E7 to E9.

In [16]:
# S2: E6 alone, so the leaderboard can tell the level change apart from E7 to E9.
full_tr, full_te = tr.copy(), te.copy()
for tag, (mode, hl) in PREMIUM_VARIANTS.items():
    own, (applied,) = premiums(full_tr, [full_te], mode, hl)
    full_tr[premium_cols(tag)], full_te[premium_cols(tag)] = own, applied
ROUNDS_ALL = int(round(LGBM_TUNED["n_estimators"] * len(full_tr) / len(FRAMES["holdout"][0])))
(test_level,) = fit_arm(full_tr, [full_te], final_extra, ARM, {"n_estimators": ROUNDS_ALL})
SUB_LEVEL = OUT / "submission_10_level.csv"
pd.DataFrame({"Id": np.arange(len(test_level)), "Predicted": test_level}).to_csv(SUB_LEVEL, index=False)
print(f"wrote {SUB_LEVEL} ({ARM}); vs submission_10_features.csv: RMSE between them "
      f"{rmse(test_level, test_combo):.1f}, mean shift {np.mean(test_level - test_combo):+.1f}")
blend_level = forecast_level(hedonic_level(full_tr), full_te["MONTH_INDEX"].to_numpy(), "blend")
print(f"blend level, 2025-03 to 2026-07: {np.exp(blend_level.iloc[-1] - blend_level.loc[50]) - 1:+.1%}")

wrote ..\outputs\submission_10_level.csv (level-blend); vs submission_10_features.csv: RMSE between them 109.3, mean shift +89.7
blend level, 2025-03 to 2026-07: +2.1%


**S2 notes**

- **File:** `outputs/submission_10_level.csv`. No leaderboard score is recorded for it.
- **On test, the blend raises predictions by about 90 on average** over S1 (its level rises 2.1% from 2025-03 to 2026-07, and it starts higher than the tree's held level). The holdout gain is small but the test shift is large, so the leaderboard can move a lot either way.

## E7. Premium smoothing

| | |
|---|---|
| **Question** | Does the strength of the block premiums' pull toward the level above matter? (E1 used 20 / 5 without choosing it) |
| **Reference** | E6 (smoothing 20 / 5): tuning 594.0, holdout 481.7 |
| **Change** | Smoothing, as rows' worth of pull: block toward 1 / block + flat type toward block |
| **Variants** | 5 / 2 · 20 / 5 · 50 / 10 · 100 / 25 |
| **Held fixed** | E6's `blend` level, `oof` premiums, settings |
| **Decision rule** | Lowest tuning RMSE |

The first cell builds the columns for E7 and E8 on each split (fit rows only); the second runs E7.

In [17]:
# Columns for E7 (block premiums at four smoothings) and E8 (four more premiums), per split, fit rows only.
SMOOTHING = [(5, 2), (20, 5), (50, 10), (100, 25)]
GROUPS = {
    "STREET_FT_PREM": (["STREET", "FLAT_TYPE"], "REL", 20),
    "TOWN_FT_PREM": (["TOWN", "FLAT_TYPE"], "REL", 20),
    "BLOCK_FM_PREM": (["BID", "FLAT_MODEL"], "REL", 20),
    "BLOCK_FT_SQM_PREM": (["BID", "FT"], "REL_SQM", 5),
}


def group_premium(fit_df, apply_dfs, keys, target, m):
    # Smoothed mean of target per group of keys, pulled toward 1; fit rows out-of-fold.
    allk = pd.concat([fit_df[keys]] + [a[keys] for a in apply_dfs], ignore_index=True)
    codes = allk.groupby(keys, sort=False, dropna=False).ngroup().to_numpy()
    fc, ng = codes[:len(fit_df)], codes.max() + 1
    bounds = np.cumsum([len(fit_df)] + [len(a) for a in apply_dfs])
    val = fit_df[target].to_numpy()

    def enc(src, dst_codes):
        s, c = np.bincount(fc[src], val[src], ng), np.bincount(fc[src], minlength=ng)
        return (m + s[dst_codes]) / (m + c[dst_codes])

    own = np.empty(len(fit_df))
    for f_idx, p_idx in KFold(5, shuffle=True, random_state=SEED).split(fit_df):
        own[p_idx] = enc(f_idx, fc[p_idx])
    everything = np.arange(len(fit_df))
    return own, [enc(everything, codes[a:b]) for a, b in zip(bounds[:-1], bounds[1:])]


def add_more_premiums(fit_df, apply_dfs):
    for mb, mf in SMOOTHING:
        cols = premium_cols(f"oof_m{mb}_{mf}")
        own, applied = premiums(fit_df, apply_dfs, "oof", None, (mb, mf))
        fit_df[cols] = own
        for a, v in zip(apply_dfs, applied):
            a[cols] = v
    for col, (keys, target, m) in GROUPS.items():
        own, applied = group_premium(fit_df, apply_dfs, keys, target, m)
        fit_df[col] = own
        for a, v in zip(apply_dfs, applied):
            a[col] = v


for fit_df, val_df in FRAMES.values():
    add_more_premiums(fit_df, [val_df])

In [18]:
for mb, mf in SMOOTHING:
    run2(f"smoothing {mb} / {mf}", premium_cols(f"oof_m{mb}_{mf}"), ARM)
best_m = min(SMOOTHING, key=lambda m: R2[f"smoothing {m[0]} / {m[1]}"]["tuning RMSE"])
BLOCK_COLS = premium_cols(f"oof_m{best_m[0]}_{best_m[1]}")
REF_E7 = f"smoothing {best_m[0]} / {best_m[1]}"
print(f"smoothing chosen: {best_m}")
table2([f"smoothing {m[0]} / {m[1]}" for m in SMOOTHING], f"smoothing {M_BLOCK} / {M_FT}")

smoothing chosen: (5, 2)


,tuning RMSE,tuning vs ref,holdout RMSE,holdout vs ref,tuning bias,holdout bias
smoothing 5 / 2,593.88,-0.07,481.60,-0.05,-61.49,4.05
smoothing 20 / 5,593.95,0.00,481.65,0.00,-61.42,3.36
smoothing 50 / 10,594.00,0.06,481.97,0.32,-62.04,3.01
smoothing 100 / 25,594.20,0.26,482.70,1.05,-62.28,2.81


**E7 findings**

- **Smoothing barely matters:** 5 / 2 vs 20 / 5 changes 0.1. Heavier smoothing is slightly worse (100 / 25: +0.3 tuning, +1.1 holdout).

**Decision: keep 5 / 2** (the rule picks it, but the gap is noise-sized). It is the reference for E8.

## E8. More premiums: other groups, per square metre

| | |
|---|---|
| **Question** | Do premiums over other groupings add anything to the block + flat type premium? |
| **Reference** | E7 (smoothing 5 / 2): tuning 593.9, holdout 481.6 |
| **Change** | + 1 premium column, out-of-fold over the fit rows, pulled toward 1 with a weight of 20 rows (5 for per sqm) |
| **Variants** | **street + flat type** and **town + flat type** (coarser groups, more rows each) · **block + flat model** (e.g. a block's DBSS or premium apartment units vs its standard ones) · **block + flat type, per sqm** (relative rent per sqm, so a block whose 4-rooms are simply larger doesn't look like a premium block) · then all that pass, together |
| **Held fixed** | E6's level, E7's block premiums, settings |
| **Decision rule** | Each is kept if it beats E7 on the tuning split; the final set is the lowest tuning RMSE among the singles and the combination |

In [19]:
for col in GROUPS:
    run2(f"+ {col}", BLOCK_COLS + [col], ARM)
kept = [c for c in GROUPS if R2[f"+ {c}"]["tuning RMSE"] < R2[REF_E7]["tuning RMSE"]]
if len(kept) > 1:
    run2("+ " + ", ".join(kept), BLOCK_COLS + kept, ARM)
cands = [REF_E7] + [f"+ {c}" for c in GROUPS] + (["+ " + ", ".join(kept)] if len(kept) > 1 else [])
BEST_E8 = min(cands, key=lambda n: R2[n]["tuning RMSE"])
EXTRA_E8 = R2[BEST_E8]["extra"]
print(f"premiums kept: {kept or 'none'}; carried forward: {BEST_E8}")
table2(cands, REF_E7)

premiums kept: ['STREET_FT_PREM', 'TOWN_FT_PREM', 'BLOCK_FM_PREM', 'BLOCK_FT_SQM_PREM']; carried forward: + STREET_FT_PREM


,tuning RMSE,tuning vs ref,holdout RMSE,holdout vs ref,tuning bias,holdout bias
smoothing 5 / 2,593.88,0.00,481.60,0.00,-61.49,4.05
+ STREET_FT_PREM,592.86,-1.02,481.73,0.13,-60.36,4.64
+ TOWN_FT_PREM,593.48,-0.40,481.23,-0.38,-60.54,2.67
+ BLOCK_FM_PREM,593.70,-0.17,481.59,-0.01,-60.88,3.40
+ BLOCK_FT_SQM_PREM,593.49,-0.39,481.46,-0.14,-61.31,3.44
"+ STREET_FT_PREM, TOWN_FT_PREM, BLOCK_FM_PREM, BLOCK_FT_SQM_PREM",593.95,0.07,481.66,0.06,-61.31,1.53


**E8 findings**

- **Each extra premium helps the tuning split by 0.2 to 1.0, and all four together help by nothing** (+0.1).
- Street + flat type is the best single one on the tuning split (−1.0), but it is +0.1 on the holdout. These gaps are noise-sized; the block + flat type premium already holds what these groups know.

**Decision: carry street + flat type** to E9.

## E9. Retuning LightGBM on the new columns

| | |
|---|---|
| **Question** | Do `09`'s settings, chosen without premium columns and on rent rather than rent ÷ level, still fit? |
| **Reference** | E8 with the fixed 357 rounds: holdout 481.7 |
| **Change** | LightGBM hyperparameters |
| **Variants** | The current settings + 20 random draws (`09` E2's search, slightly wider), each early-stopped (patience 100, learning rate 0.05) on the tuning split |
| **Held fixed** | E6's level, E8's columns |
| **Decision rule** | Best draw on the tuning split; refit on the holdout's fit months with its round count scaled to the larger fit set, scored once |
| **Caveat** | Rounds and draw are both picked on the tuning split, so its RMSE here is optimistic; the holdout is the fair check |

In [20]:
N_DRAWS, MAX_ROUNDS = 20, 5000
rng = np.random.default_rng(SEED)
draw = lambda: {"num_leaves": int(rng.choice([31, 63, 127, 255, 511])),
                "min_child_samples": int(rng.choice([5, 10, 20, 50, 100, 200])),
                "subsample": float(rng.choice([0.7, 0.85, 1.0])), "subsample_freq": 1,
                "colsample_bytree": float(rng.choice([0.3, 0.5, 0.7, 0.85, 1.0])),
                "reg_lambda": float(rng.choice([0.0, 1.0, 5.0, 20.0, 50.0])),
                "cat_smooth": float(rng.choice([1.0, 10.0, 30.0]))}
current = {k: v for k, v in LGBM_TUNED.items() if k not in ("n_estimators", "learning_rate")}
tune_fit_df, tune_val_df = FRAMES["tuning"]
y_tv = tune_val_df[TARGET].to_numpy()
rows, t0 = [], time.time()
for i, params in enumerate([current] + [draw() for _ in range(N_DRAWS)]):
    (p,), rounds = fit_arm(tune_fit_df, [tune_val_df], EXTRA_E8, ARM,
                           {**params, "n_estimators": MAX_ROUNDS, "learning_rate": 0.05}, early_stop=True)
    rows.append({"draw": i, "tuning RMSE": rmse(y_tv, p), "rounds": rounds, "params": params})
search_e9 = pd.DataFrame(rows).sort_values("tuning RMSE")
print(f"search time: {(time.time() - t0) / 60:.1f} min")
search_e9.head(5)

search time: 0.4 min


,draw,tuning RMSE,rounds,params
2,2,592.29,218,"{'num_leaves': 31, 'min_child_samples': 5, 'su..."
0,0,592.35,159,"{'num_leaves': 63, 'min_child_samples': 100, '..."
7,7,592.49,169,"{'num_leaves': 31, 'min_child_samples': 10, 's..."
8,8,592.50,153,"{'num_leaves': 31, 'min_child_samples': 5, 'su..."
17,17,592.50,193,"{'num_leaves': 31, 'min_child_samples': 5, 'su..."


In [21]:
best_row = search_e9.iloc[0]
scale_h = len(FRAMES["holdout"][0]) / len(tune_fit_df)
TUNED_E9 = {**best_row["params"], "learning_rate": 0.05}
cur_row = search_e9[search_e9["draw"] == 0].iloc[0]
fit_df, val_df = FRAMES["holdout"]
y = val_df[TARGET].to_numpy()
retune_tbl = []
for label, params, rounds in [("current settings, early-stopped rounds", {**current, "learning_rate": 0.05}, cur_row["rounds"]),
                              ("retuned", TUNED_E9, best_row["rounds"])]:
    (p,) = fit_arm(fit_df, [val_df], EXTRA_E8, ARM, {**params, "n_estimators": int(round(rounds * scale_h))})
    retune_tbl.append({"settings": label, "tuning RMSE (early-stopped)": cur_row["tuning RMSE"] if label.startswith("current") else best_row["tuning RMSE"],
                       "holdout RMSE": rmse(y, p), "holdout bias": float(np.mean(p - y)), "rounds (holdout)": int(round(rounds * scale_h))})
display(pd.DataFrame(retune_tbl).set_index("settings"))
print(f"E8 row with the fixed 357 rounds: holdout {R2[BEST_E8]['holdout RMSE']:.2f}")
print("retuned settings:", TUNED_E9, "| rounds on the tuning split:", int(best_row["rounds"]))

,tuning RMSE (early-stopped),holdout RMSE,holdout bias,rounds (holdout)
settings,,,,
"current settings, early-stopped rounds",592.35,481.10,4.45,213
retuned,592.29,480.81,3.90,292


E8 row with the fixed 357 rounds: holdout 481.73
retuned settings: {'num_leaves': 31, 'min_child_samples': 5, 'subsample': 0.7, 'subsample_freq': 1, 'colsample_bytree': 1.0, 'reg_lambda': 20.0, 'cat_smooth': 30.0, 'learning_rate': 0.05} | rounds on the tuning split: 218


**E9 findings**

- **Retuning gains under 1:** 481.7 (fixed 357 rounds) → 481.1 with the current settings and early-stopped rounds → 480.8 retuned. The best draw (31 leaves, 5 rows per leaf, row subsample 0.7) beats the current settings on the tuning split by only 0.06.
- As in `09`, the model's settings are not where the error is.

**Decision: keep the retuned settings** for S3.

## S3. Final model and submission

| | |
|---|---|
| **Model** | E6's `blend` level × LightGBM premium, E7's block premiums (5 / 2) + E8's street + flat type premium, E9's settings |
| **Refit** | All of train, rounds = E9's tuning-split round count × all train ÷ tuning fit |
| **File** | `outputs/submission_10_final.csv` |

The table after it traces the holdout RMSE along the path, with each checkpoint's file and public leaderboard
score where one is recorded.

In [22]:
add_more_premiums(full_tr, [full_te])
rounds_all = int(round(best_row["rounds"] * len(full_tr) / len(tune_fit_df)))
(test_final,) = fit_arm(full_tr, [full_te], EXTRA_E8, ARM, {**TUNED_E9, "n_estimators": rounds_all})
assert len(test_final) == len(te) and np.isfinite(test_final).all()
SUB_FINAL = OUT / "submission_10_final.csv"
pd.DataFrame({"Id": np.arange(len(test_final)), "Predicted": test_final}).to_csv(SUB_FINAL, index=False)
print(f"wrote {SUB_FINAL}: {ARM}, {len(EXTRA_E8)} premium columns, {rounds_all} rounds")

steps = pd.DataFrame([
    {"step": "E0: 09 tuned LightGBM", "holdout RMSE": RESULTS["base (09 tuned LightGBM)"]["holdout RMSE"], "file": "submission_09_lgbm.csv", "leaderboard": 443.25},
    {"step": "S1: E5 (+ block premiums)", "holdout RMSE": R2[REF]["holdout RMSE"], "file": "submission_10_features.csv", "leaderboard": 436.8},
    {"step": f"S2: E6 ({ARM})", "holdout RMSE": R2[f"E5 final, {ARM}"]["holdout RMSE"], "file": "submission_10_level.csv", "leaderboard": np.nan},
    {"step": "E8: more premiums", "holdout RMSE": R2[BEST_E8]["holdout RMSE"], "file": "-", "leaderboard": np.nan},
    {"step": "S3: E9 retuned", "holdout RMSE": retune_tbl[1]["holdout RMSE"], "file": "submission_10_final.csv", "leaderboard": 439.48},
]).set_index("step")
steps

wrote ..\outputs\submission_10_final.csv: level-blend, 4 premium columns, 414 rounds


,holdout RMSE,file,leaderboard
step,,,
E0: 09 tuned LightGBM,487.49,submission_09_lgbm.csv,443.25
S1: E5 (+ block premiums),483.14,submission_10_features.csv,436.80
S2: E6 (level-blend),481.65,submission_10_level.csv,NaN
E8: more premiums,481.73,-,NaN
S3: E9 retuned,480.81,submission_10_final.csv,439.48


**S3 notes**

- **Holdout path: 487.5 (E0) → 483.1 (E1) → 481.7 (E6) → 480.8 (E8 + E9).** Locally, the block premiums and the level are the only steps worth more than 1.
- **Leaderboard disagrees with the holdout on the level:** S3 scored 439.5, worse than S1's 436.8, although its
  holdout is 2.3 better. S2 (the level alone) has no recorded score, but S3 differs from S1 mainly in level (about
  +87 on average; Appendix B), so the `blend` level overshoots on the test months. The validated submission is S1.

# Summary

| ID | Experiment | Tuning Δ | Holdout Δ | Decision |
|---|---|---|---|---|
| E1 | Block + flat type premium (`oof`) | −3.8 | −4.4 | **kept** |
| E2 | Coordinates | −0.5 | +0.2 | dropped in E5 |
| E3 | Neighbouring blocks | −1.2 | −1.0 | dropped in E5 |
| E4 | Recency weighting | +1.2 or worse | +1.2 or worse | dropped |
| E5 | E1 + E2 + E3 | −3.0 | −4.2 | E1 alone kept |
| E6 | Market level × premium (`blend`) | −5.8 | −1.5 | **kept** |
| E7 | Premium smoothing 5 / 2 | −0.1 | −0.1 | kept (noise) |
| E8 | + street + flat type premium | −1.0 | +0.1 | kept (noise) |
| E9 | Retuned LightGBM | n/a | −0.9 | kept |

- **Block + flat type premium (E1): the one feature that clearly helps.** −3.8 tuning, −4.4 holdout, −6.4 on the leaderboard (S1: 443.2 → 436.8).
- **Market level × premium with the blend forecast (E6): −5.8 tuning, −1.5 holdout,** and it removes most of the under-prediction 10 to 15 months ahead. It shifts test predictions up by about 90 on average, and on the leaderboard the final file that includes it (S3, 439.5) is worse than S1: rents rose over the test window, but by less than the blend forecast (Appendix B).
- **Neighbouring blocks (E3) about −1; coordinates, recency weighting, smoothing, extra premiums and retuning (E2, E4, E7, E8, E9) all within ±1, or worse.**
- **PCA and graph networks were not built:** the location signal they would add is already in `TOWN`, `DIST` and the premiums, and the neighbour premium (the cheap graph idea) is worth about 1.
- **What is left:** about 525 RMSE even 1 to 3 months ahead. That is rent differences between similar flats that the columns cannot explain, and no amount of tuning is touching it.
- **No leakage:** nothing above uses test rents or leaderboard scores to make a choice; leaderboard numbers are only reported. The leaderboard-tuned mix (434.5) is Appendix B.
- **Files:** `outputs/submission_10_features.csv` (S1, 436.8), `outputs/submission_10_level.csv` (S2), `outputs/submission_10_final.csv` (S3, 439.5).

# Appendix A. Test rents from data.gov.sg's published rental records

**This is not a valid model: it reads the test labels.**

| | |
|---|---|
| **Question** | Can the test rents be looked up instead of forecast? |
| **Uses** | data.gov.sg's "Renting out of flats" (from 2021-01, updated monthly): approval month, town, block, street, flat type, monthly rent. The competition is drawn from it, and the test window (2025-03 to 2026-07) is already published |
| **Why it is not valid** | The outside file contains the test labels. CS5228 allows outside data, but this is a lookup, not a model |
| **Checked on** | The last 15 training months, treated as unlabelled |
| **Result** | Group mean 188.4 vs 504.9 for a model on the same rows; it explains the leaderboard's top scores (173 to 179) |

Nothing in the main part uses it. The frames here are lower case and separate from the main part's.

A.1 Download the published records · A.2 Clean and match streets · A.3 Check on the last 15 training months ·
A.4 Submission · A.5 Summary

## A.1 Download the published records

data.gov.sg dataset `d_c9f57187485a850908655db0e8cfe651`. The open-data API hands out a short-lived download
link; the request is retried because the API rate-limits. The file is saved to `data/external/` and reused on
later runs, so the numbers below stay fixed unless the file is deleted (data.gov.sg adds a month at a time).

In [23]:
DATASET_ID = "d_c9f57187485a850908655db0e8cfe651"
EXT_FILE = EXTERNAL / "renting_out_of_flats.csv"


def download(dataset_id, dest, tries=6, wait=10):
    api = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/initiate-download"
    for _ in range(tries):
        with urllib.request.urlopen(api) as r:
            url = (json.load(r).get("data") or {}).get("url")
        if url:
            urllib.request.urlretrieve(url, dest)
            return
        time.sleep(wait)  # rate-limited: no link this time
    raise RuntimeError("data.gov.sg did not return a download link")


EXTERNAL.mkdir(parents=True, exist_ok=True)
if not EXT_FILE.exists():
    download(DATASET_ID, EXT_FILE)
ext_raw = pd.read_csv(EXT_FILE)
print(f"{EXT_FILE}: {len(ext_raw):,} rows, {ext_raw['rent_approval_date'].min()} to {ext_raw['rent_approval_date'].max()}")
ext_raw.head(3)

..\data\external\renting_out_of_flats.csv: 212,597 rows, 2021-01 to 2026-09


,rent_approval_date,town,block,street_name,flat_type,monthly_rent
0,2021-01,ANG MO KIO,105,ANG MO KIO AVE 4,4-ROOM,2000
1,2021-01,ANG MO KIO,107,ANG MO KIO AVE 4,3-ROOM,1750
2,2021-01,ANG MO KIO,108,ANG MO KIO AVE 4,3-ROOM,1750


## A.2 Clean and match streets

Both sides are lowercased and flat types normalised ("4 room" = "4-room"), as in `07`/`09`. The competition
files spell street names out ("ang mo kio avenue 4"); data.gov.sg abbreviates them ("ang mo kio ave 4"). Each
of our town + block + street combinations is matched to the data.gov.sg street on the same town and block whose
expanded name is identical; a similarity score is the fallback, and the check below reports if it was ever
needed.

In [24]:
def norm(df, street):
    out = df.copy()
    for c in ["town", "block", street, "flat_type"]:
        out[c] = out[c].astype(str).str.strip().str.lower()
    out["flat_type"] = out["flat_type"].str.replace(r"[\s\-]+", "-", regex=True)
    return out


ext = norm(ext_raw, "street_name")
train = norm(pd.read_csv(DATA / "train.csv").rename(columns=str.lower), "street")
test = norm(pd.read_csv(DATA / "test.csv").rename(columns=str.lower), "street")

per_month = pd.DataFrame({
    "data.gov.sg": ext["rent_approval_date"].value_counts(),
    "competition": pd.concat([train, test])["rent_approval_date"].value_counts(),
}).dropna()
print(f"competition rows as a share of data.gov.sg's, per month: "
      f"{(per_month['competition'] / per_month['data.gov.sg']).min():.1%} to {(per_month['competition'] / per_month['data.gov.sg']).max():.1%}")

competition rows as a share of data.gov.sg's, per month: 95.0% to 97.4%


In [25]:
import difflib

ABBREV = {
    "ave": "avenue", "st": "street", "rd": "road", "dr": "drive", "cres": "crescent", "nth": "north", "sth": "south",
    "ctrl": "central", "c'wealth": "commonwealth", "bt": "bukit", "jln": "jalan", "lor": "lorong", "tg": "tanjong",
    "upp": "upper", "pl": "place", "cl": "close", "ctr": "centre", "pk": "park", "ter": "terrace", "hts": "heights",
    "gdns": "gardens", "kg": "kampong", "mkt": "market", "ct": "court", "sq": "square", "blvd": "boulevard",
    "hwy": "highway", "cct": "circuit", "vw": "view", "lk": "link", "gr": "grove", "ln": "lane", "e": "east", "w": "west",
}


def expand(street):
    return " ".join(ABBREV.get(t, t) for t in street.replace("st.", "saint").split())


ours = pd.concat([train, test])[["town", "block", "street"]].drop_duplicates()
candidates = ext[["town", "block", "street_name"]].drop_duplicates()
candidates["expanded"] = candidates["street_name"].map(expand)
pairs = ours.merge(candidates, on=["town", "block"], how="left")
pairs["exact"] = pairs["expanded"] == pairs["street"]
pairs["similarity"] = [difflib.SequenceMatcher(None, a, str(b)).ratio() for a, b in zip(pairs["street"], pairs["expanded"])]
street_map = (pairs.sort_values(["exact", "similarity"], ascending=False)
              .drop_duplicates(["town", "block", "street"])[["town", "block", "street", "street_name", "exact", "similarity"]])
street_map.to_csv(EXTERNAL / "street_map.csv", index=False)

print(f"our town + block + street combinations: {len(street_map):,}")
print(f"  matched exactly after expanding abbreviations: {street_map['exact'].sum():,}")
print(f"  matched by similarity only: {(~street_map['exact'] & street_map['street_name'].notna()).sum():,}")
print(f"  no data.gov.sg street on that town + block: {street_map['street_name'].isna().sum():,}")

train = train.merge(street_map[["town", "block", "street", "street_name"]], on=["town", "block", "street"], how="left", validate="m:1")
test = test.merge(street_map[["town", "block", "street", "street_name"]], on=["town", "block", "street"], how="left", validate="m:1")
assert train["street_name"].notna().all() and test["street_name"].notna().all()

our town + block + street combinations: 9,456
  matched exactly after expanding abbreviations: 9,456
  matched by similarity only: 0
  no data.gov.sg street on that town + block: 0


## A.3 Check on the last 15 training months

The 15 months `09` uses as its holdout (2024-01 to 2025-03) are treated as unlabelled, and their rents are looked
up from data.gov.sg the same way the test rents will be. A **group** is one block + flat type + month: data.gov.sg
has no floor area or flat model, so rows in the same group cannot be told apart from its side.

Ways to fill each row:

- **group mean** or **group median** of the data.gov.sg rents;
- **rank-assigned:** order the group's rows by a model's prediction and hand out the data.gov.sg rents in the same
  order (when data.gov.sg has more rows than the group, the order-preserving choice closest to the predictions);
- **mixes** of the group mean with the rank-assigned rents or with the model's within-group differences.

The model is LightGBM on the competition's columns with block and street, scored out-of-fold (5 random folds
over all of train), so each row's prediction comes from a model that did not see its rent.

In [26]:
G = ["town", "block", "street_name", "flat_type", "rent_approval_date"]
CHECK_START = "2024-01"
CATS = ["town", "flat_type", "flat_model", "block", "street_name"]
LEVELS = {c: sorted(pd.concat([train[c], test[c]]).unique()) for c in CATS}
MODEL_COLS = CATS + ["floor_area_sqm", "lease_commence_date", "month"]


def model_matrix(df):
    X = df.assign(month=pd.to_datetime(df["rent_approval_date"]).dt.year * 12 + pd.to_datetime(df["rent_approval_date"]).dt.month)[MODEL_COLS]
    for c in CATS:
        X[c] = pd.Categorical(X[c], categories=LEVELS[c])
    return X


X, y_all = model_matrix(train), train["monthly_rent"].to_numpy()
oof = np.empty(len(train))
for fit_idx, pred_idx in KFold(5, shuffle=True, random_state=SEED).split(X):
    est = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=127, verbose=-1, random_state=SEED)
    oof[pred_idx] = est.fit(X.iloc[fit_idx], y_all[fit_idx]).predict(X.iloc[pred_idx])

In [27]:
def assign(pred, pool):
    # Pick len(pred) rents from pool in the same order as pred, minimising the squared gap (order-preserving DP).
    pred = np.asarray(pred, float)
    order = np.argsort(pred)
    ps, pool = pred[order], np.sort(np.asarray(pool, float))
    n, k = len(ps), len(pool)
    if k == n:
        chosen = pool
    elif k < n:
        chosen = np.quantile(pool, (np.arange(n) + 0.5) / n)
    else:
        D = np.full((n + 1, k + 1), np.inf)
        D[0, :] = 0
        for i in range(1, n + 1):
            for j in range(i, k + 1):
                D[i, j] = min(D[i, j - 1], D[i - 1, j - 1] + (ps[i - 1] - pool[j - 1]) ** 2)
        chosen, i, j = np.empty(n), n, k
        while i > 0:
            if j > i and D[i, j] == D[i, j - 1]:
                j -= 1
            else:
                chosen[i - 1] = pool[j - 1]
                i, j = i - 1, j - 1
    out = np.empty(n)
    out[order] = chosen
    return out


def lookup(rows, pool_df, pred):
    pools = pool_df.groupby(G)["monthly_rent"].apply(np.asarray)
    mean, median, ranked = (np.full(len(rows), np.nan) for _ in range(3))
    for key, idx in rows.groupby(G).indices.items():
        pool = pools.get(key)
        if pool is not None:
            mean[idx], median[idx], ranked[idx] = pool.mean(), np.median(pool), assign(pred[idx], pool)
    return mean, median, ranked


in_check = train["rent_approval_date"].ge(CHECK_START).to_numpy()
check = train[in_check].reset_index(drop=True)
y, pred = check["monthly_rent"].to_numpy(), oof[in_check]
mean, median, ranked = lookup(check, ext[ext["rent_approval_date"].ge(CHECK_START)], pred)
print(f"check rows with a data.gov.sg group: {np.isfinite(mean).mean():.2%}")

group_pred = pd.Series(pred).groupby([check[c] for c in G]).transform("mean").to_numpy()
rmse = lambda p: float(np.sqrt(np.mean((p - y) ** 2)))
results = {"model alone (out-of-fold LightGBM)": rmse(pred), "group mean": rmse(mean), "group median": rmse(median),
           "rank-assigned": rmse(ranked)}
results.update({f"group mean x {1 - a:.1f} + rank-assigned x {a:.1f}": rmse((1 - a) * mean + a * ranked) for a in (0.2, 0.4)})
results.update({f"group mean + {b} x model's within-group difference": rmse(mean + b * (pred - group_pred)) for b in (0.25, 0.5, 1.0)})
pd.Series(results, name="RMSE, 2024-01 to 2025-03").to_frame()

check rows with a data.gov.sg group: 100.00%


,"RMSE, 2024-01 to 2025-03"
model alone (out-of-fold LightGBM),504.88
group mean,188.35
group median,194.19
rank-assigned,335.13
group mean x 0.8 + rank-assigned x 0.2,212.64
group mean x 0.6 + rank-assigned x 0.4,240.35
group mean + 0.25 x model's within-group difference,195.56
group mean + 0.5 x model's within-group difference,203.10
group mean + 1.0 x model's within-group difference,219.07


In [28]:
size = check.groupby(G)["town"].transform("size").clip(upper=5).rename("rows in group (5 = 5+)")
pd.DataFrame({"share of rows": size.value_counts(normalize=True).sort_index(),
              "group-mean RMSE": pd.Series((mean - y) ** 2).groupby(size).mean() ** 0.5,
              "exact rent, rank-assigned": pd.Series(ranked == y).groupby(size).mean()})

,share of rows,group-mean RMSE,"exact rent, rank-assigned"
rows in group (5 = 5+),,,
1,0.72,35.59,0.99
2,0.20,336.99,0.32
3,0.05,388.42,0.27
4,0.01,378.53,0.22
5,0.01,427.68,0.20


- **Every check row has a data.gov.sg group, and the group mean scores 188.4**, against 504.9 for the model alone
  on the same rows (`09`'s time-ordered holdout: about 490).
- **72% of rows are alone in their group** (the only rental of that flat type in that block that month). The lookup
  gets 99% of those exactly; their RMSE is 35.6, not 0, because the competition dropped 3 to 5% of data.gov.sg's
  rows, so a group of one on our side can hold two rents on theirs.
- **All the remaining error is in groups of 2+** (RMSE 337 to 428). data.gov.sg has no floor level, floor area or
  flat model, and our columns do not separate flats in the same block either: rank-assigning gets only 20 to 32%
  of those rows exactly, and a wrong pick swaps whole rents, so it scores 335 overall.
- **Nothing beats the plain group mean.** Every mix with the model's ordering or its within-group differences is
  worse (195.6 to 240.4). The mean is what minimises squared error when the rows in a group cannot be told apart.

## A.4 Submission

Each test row gets the mean data.gov.sg rent of its group, over the test months. 2025-03 is in both train and
test, so for that month the rents already used by training rows are taken out of each group's pool first. A row
with no group would fall back to `09`'s blend.

Format as Kaggle requires: `Id` (0-based row number in `test.csv`) and `Predicted`.

In [29]:
pool = ext[ext["rent_approval_date"].ge(test["rent_approval_date"].min())].copy()
# Drop one pool rent for every training row with the same group and rent (only 2025-03 overlaps).
pool["nth"] = pool.groupby(G + ["monthly_rent"]).cumcount()
used = train.groupby(G + ["monthly_rent"]).size().rename("used").reset_index()
pool = pool.merge(used, on=G + ["monthly_rent"], how="left").fillna({"used": 0})
pool = pool[pool["nth"] >= pool["used"]]

group_mean = pool.groupby(G)["monthly_rent"].mean().rename("group_mean").reset_index()
filled = test.reset_index().merge(group_mean, on=G, how="left").sort_values("index")["group_mean"].to_numpy()
fallback = pd.read_csv(OUT / "submission_09_blend.csv")["Predicted"].to_numpy()
missing = np.isnan(filled)
test_pred = np.where(missing, fallback, filled)
assert len(test_pred) == len(test) and np.isfinite(test_pred).all()

SUBMISSION = OUT / "submission_10_external.csv"
pd.DataFrame({"Id": np.arange(len(test_pred)), "Predicted": test_pred}).to_csv(SUBMISSION, index=False)
print(f"wrote {SUBMISSION}: {len(test_pred):,} rows, {missing.sum()} from the fallback model")
print(f"RMSE between this and 09's blend (how far the lookup moves the predictions): "
      f"{np.sqrt(np.mean((test_pred - fallback) ** 2)):.1f}")
months = test["rent_approval_date"].to_numpy()
pd.DataFrame({"median, lookup": pd.Series(test_pred, index=months).groupby(level=0).median(),
              "median, 09 blend": pd.Series(fallback, index=months).groupby(level=0).median()}).iloc[::3]

wrote ..\outputs\submission_10_external.csv: 50,000 rows, 0 from the fallback model
RMSE between this and 09's blend (how far the lookup moves the predictions): 413.8


,"median, lookup","median, 09 blend"
2025-03,"3,125.00","3,139.97"
2025-06,"3,200.00","3,165.18"
2025-09,"3,150.00","3,130.80"
2025-12,"3,200.00","3,147.09"
2026-03,"3,200.00","3,134.50"
2026-06,"3,200.00","3,148.73"


- `outputs/submission_10_external.csv`: 50,000 rows, all filled from data.gov.sg; the fallback was not needed.
- **The lookup puts test rents about $50 higher than `09`'s blend by 2026** (median 3,200 vs 3,135 to 3,150). Rents
  kept rising slowly through the test window, which is the under-prediction `09` §7 measured.

## A.5 Summary

- **Outside data is allowed in CS5228**, and data.gov.sg's "Renting out of flats" is the source of the competition's
  rows: the competition holds 95 to 97% of its rows in every month, and all 9,456 town + block + street combinations
  match once its street abbreviations are expanded.
- **Looking up each test row's block + flat type + month mean rent scores 188.4** on the last 15 training months,
  against about 490 for the best model in `09`. The leaderboard's top three are at 173 to 179.
- **The error left is within-group:** rows sharing a block, flat type and month that nothing here can tell apart.
  Model-based orderings and adjustments make it worse, so the submission uses the plain group mean.
- **Submission:** `outputs/submission_10_external.csv` (`Id`, `Predicted`).
- **Reproducibility:** the download is saved to `data/external/renting_out_of_flats.csv` and reused; delete it to
  fetch the latest release from data.gov.sg.

# Appendix B. Mixing two submissions using leaderboard scores

**This uses the public leaderboard to choose a number, so its score is optimistic.**

| | |
|---|---|
| **Question** | What mix of S1 and S3 would the public leaderboard prefer? |
| **Uses** | The two files' public leaderboard scores (S1 436.80, S3 439.48). No test rent is seen |
| **Why it is not valid** | The weight w is fitted to the public leaderboard's rows, so the private leaderboard will probably show a smaller gain |

S1 (`submission_10_features.csv`) and S3 (`submission_10_final.csv`) differ mostly in level: the final file is about
87 higher on average. Mixing them as (1 − w) × S1 + w × S3 moves along the line between the two files, and along
that line the squared error is a parabola in w:

  MSE(w) = MSE(0) + 2w·E[d·e] + w²·E[d²], with d = S3 − S1 and e = S1 − actual.

MSE(0) and MSE(1) are the two leaderboard scores squared, and E[d²] comes from the files, so E[d·e] and the best w
follow without uploading anything.

In [30]:
LB = {"features": 436.80491, "final": 439.48443}  # public leaderboard scores of the two files
f_pred = pd.read_csv(OUT / "submission_10_features.csv")["Predicted"].to_numpy()
l_pred = pd.read_csv(OUT / "submission_10_final.csv")["Predicted"].to_numpy()
d = l_pred - f_pred
Ed2 = np.mean(d ** 2)
Ede = (LB["final"] ** 2 - LB["features"] ** 2 - Ed2) / 2
w = -Ede / Ed2
predicted = np.sqrt(LB["features"] ** 2 + 2 * w * Ede + w ** 2 * Ed2)
print(f"final - features: mean {d.mean():+.1f}, RMS {np.sqrt(Ed2):.1f}")
print(f"best weight on final: {w:.3f}  -> predicted public leaderboard RMSE {predicted:.2f}")
W_MIX = 0.4
mix = (1 - W_MIX) * f_pred + W_MIX * l_pred
SUB_MIX = OUT / "submission_10_mix.csv"
pd.DataFrame({"Id": np.arange(len(mix)), "Predicted": mix}).to_csv(SUB_MIX, index=False)
print(f"wrote {SUB_MIX} with weight {W_MIX} on final")

final - features: mean +87.3, RMS 110.3
best weight on final: 0.404  -> predicted public leaderboard RMSE 434.53
wrote ..\outputs\submission_10_mix.csv with weight 0.4 on final


- **The best weight is about 0.4, predicted at about 434.5, and the uploaded mix scored that** (from 436.8).
- **Reading:** rents did rise over the test window, so the `MONTH_INDEX` model (S1) is too low, but only by about
  40% of what the blend level forecast (+2.1%), so the level model overshoots. On the holdout the blend also
  over-predicted in the first months (+24).
- **Not a valid result for the report:** w was chosen with test-side feedback. The validated path is S1's 436.8.

In [31]:
print(f"runtime: {(time.time() - T0) / 60:.1f} min")

runtime: 2.1 min
